In [1]:
import re, glob
import pandas as pd
from google.colab import files

# ── 1. Upload the six WoS plain-text export files (savedrecs*.txt) ──
uploaded = files.upload()

# ── 2. Parse the WoS plain-text format ──
recs = []
for f in uploaded.keys():
    txt = open(f, encoding='utf-8-sig', errors='replace').read()
    for chunk in re.split(r'\nER\s*\n', txt):          # records end with 'ER'
        ut = re.search(r'^UT (.+)$', chunk, re.M)
        if not ut:
            continue
        def fld(tag):
            m = re.search(rf'^{tag} (.+)$', chunk, re.M)
            return m.group(1).strip() if m else None
        # titles can wrap across lines (continuations start with 3 spaces)
        tim = re.search(r'^TI (.+(?:\n   .+)*)', chunk, re.M)
        ti = ' '.join(l.strip() for l in tim.group(1).split('\n')) if tim else None
        recs.append({'ut': ut.group(1).strip(), 'title': ti,
                     'source_title': fld('SO'), 'year': fld('PY'),
                     'doi': fld('DI'), 'total_citations_wos_core': fld('TC')})

df = pd.DataFrame(recs)
print('Records:', len(df), '| unique UT:', df.ut.nunique())   # expect 2506 / 2506
df['total_citations_wos_core'] = pd.to_numeric(df.total_citations_wos_core, errors='coerce')
print('Total citations (WoS Core):', int(df.total_citations_wos_core.sum()))  # expect 56,449
print('Year range:', df.year.dropna().astype(int).min(), '-', df.year.dropna().astype(int).max())

# ── 3. Write the two repo files ──
df['ut'].to_csv('wos_corpus_UT_list.txt', index=False, header=False)
df[['title','source_title','year','doi','total_citations_wos_core']] \
    .to_csv('wos_corpus_feb2025.csv', index=False)

files.download('wos_corpus_UT_list.txt')
files.download('wos_corpus_feb2025.csv')

Saving savedrecs (1).txt to savedrecs (1).txt
Saving savedrecs (2).txt to savedrecs (2).txt
Saving savedrecs (3).txt to savedrecs (3).txt
Saving savedrecs (4).txt to savedrecs (4).txt
Saving savedrecs (5).txt to savedrecs (5).txt
Saving savedrecs.txt to savedrecs.txt
Records: 2506 | unique UT: 2506
Total citations (WoS Core): 56449
Year range: 1943 - 2024


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>